In [20]:
# =============================================================================
# COMPLETE MBTI PERSONALITY PREDICTION PIPELINE
# =============================================================================
# This is a FULL END-TO-END machine learning pipeline that:
# 1. Loads MBTI dataset
# 2. Preprocesses & cleans text
# 3. Extracts features
# 4. Trains 4 different models
# 5. Evaluates with multiple metrics
# 6. Creates visualizations
# 7. Makes predictions on new text
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import time
import pickle
import os

from nltk.corpus import stopwords
import nltk

# Download required NLTK data
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# =============================================================================
# SKLEARN IMPORTS
# =============================================================================

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

from sklearn.utils.class_weight import compute_sample_weight

# =============================================================================
# XGBOOST (Optional)
# =============================================================================

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ XGBoost not installed. Install with: pip install xgboost")

# =============================================================================
# CONFIG & SETTINGS
# =============================================================================

# Dataset path - use direct GitHub link (no local file needed)
CSV_PATH = r"C:\Users\Ateeb\-MBTI-Myers-Briggs-Personality-Type-Dataset-master\-MBTI-Myers-Briggs-Personality-Type-Dataset-master/mbti_1.csv"

# Train-test split
TEST_SIZE = 0.20
RANDOM_STATE = 42

# Model settings
ACCURACY_TARGET = 0.65
MAX_FEATURES = 5000
NGRAM_RANGE = (1, 2)

# Output directory
OUTPUT_DIR = "."

# =============================================================================
# HEADER
# =============================================================================

print("\n" + "=" * 100)
print(" " * 20 + "COMPLETE MBTI PERSONALITY PREDICTION PIPELINE")
print("=" * 100)
print("""
This pipeline includes:
  ✓ Data loading & validation
  ✓ Advanced text preprocessing
  ✓ Feature extraction (TF-IDF)
  ✓ Model training (5 models)
  ✓ Comprehensive evaluation
  ✓ Visualizations
  ✓ Prediction on new text
""")

# =============================================================================
# PHASE 1 · LOAD DATASET
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 1 · LOAD DATASET")
print("=" * 100)

print(f"\n📥 Loading dataset from:")
print(f"   {CSV_PATH}")

try:
    df = pd.read_csv(CSV_PATH, encoding="utf-8", engine="python")
    print("\n✔ Dataset downloaded successfully!")
except UnicodeDecodeError:
    print("⚠ UTF-8 failed, trying latin1...")
    df = pd.read_csv(CSV_PATH, encoding="latin1", engine="python")
    print("✔ Loaded with latin1 encoding")
except Exception as e:
    print(f"❌ Failed to load: {e}")
    raise

# Validate
print(f"\n✔ Total samples: {len(df):,}")
print(f"✔ Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3))

# Check for required columns
if 'type' not in df.columns or 'posts' not in df.columns:
    raise ValueError("Dataset must have 'type' and 'posts' columns")

# Initial stats
print(f"\n✔ Memory usage: {df.memory_usage().sum() / 1024**2:.2f} MB")

# Remove nulls
df = df.dropna(subset=['type', 'posts']).reset_index(drop=True)
print(f"✔ After removing nulls: {len(df):,} samples")

# Show distribution
print("\n📊 MBTI Type Distribution:\n")
dist = df['type'].value_counts()
total = len(df)

for mbti_type, count in dist.items():
    pct = (count / total) * 100
    bar_length = int(pct / 2)
    bar = "█" * bar_length
    print(f"  {mbti_type:6s} {count:5d} ({pct:5.1f}%) {bar}")

print(f"\n  Total: {total:,} samples")
print(f"  Classes: {len(dist)} unique types")

# =============================================================================
# PHASE 2 · ADVANCED TEXT PREPROCESSING
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 2 · ADVANCED TEXT PREPROCESSING")
print("=" * 100)

# Get stop words
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Comprehensive text preprocessing pipeline
    """
    
    if not isinstance(text, str):
        return ""
    
    # Step 1: Combine multiple posts (split by |||)
    text = text.replace("|||", " ")
    
    # Step 2: Convert to lowercase
    text = text.lower()
    
    # Step 3: Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Step 4: Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Step 5: Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)
    
    # Step 6: Replace text emojis with 'emoji' word
    emoji_patterns = [
        r':\)',  r':\(',  r':D',  r':d',  r':\/',  r':\|',
        r';D',   r';d',   r';P',  r';p',  r'XD',   r'xD',
        r'=D',   r'=\)',  r'=\(', r'=\/', r'=\|'
    ]
    for pattern in emoji_patterns:
        text = re.sub(pattern, ' emoji ', text)
    
    # Step 7: Replace punctuation marks with words
    text = re.sub(r'!+', ' exclamation ', text)
    text = re.sub(r'\?+', ' question ', text)
    text = re.sub(r'\.{2,}', ' ellipsis ', text)
    text = re.sub(r'#+', ' hashtag ', text)
    
    # Step 8: Remove remaining punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Step 9: Remove digits
    text = re.sub(r'\d+', '', text)
    
    # Step 10: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 11: Remove stop words
    words = text.split()
    text = ' '.join([word for word in words if word not in stop_words and len(word) > 2])
    
    return text

print("\n🧹 Preprocessing text (this may take 1-2 minutes)...")
start_time = time.time()

# Apply preprocessing
df['cleaned'] = df['posts'].apply(preprocess_text)

# Remove empty texts
df = df[df['cleaned'].str.len() > 0].reset_index(drop=True)

end_time = time.time()

print(f"✔ Preprocessing completed in {(end_time - start_time):.2f} seconds")
print(f"✔ After removing empty texts: {len(df):,} samples")

# Show examples
print("\n📝 Sample Cleaned Text:\n")
for i in range(3):
    print(f"Sample {i+1}:")
    print(f"  Original: {df['posts'].iloc[i][:100]}...")
    print(f"  Cleaned:  {df['cleaned'].iloc[i][:100]}...")
    print()

# Text length statistics
print("📊 Text Statistics:")
lengths = df['cleaned'].str.split().str.len()
print(f"  Min words: {lengths.min()}")
print(f"  Max words: {lengths.max()}")
print(f"  Mean words: {lengths.mean():.1f}")
print(f"  Median words: {lengths.median():.1f}")

# =============================================================================
# PHASE 3 · TRAIN-TEST SPLIT
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 3 · TRAIN-TEST SPLIT")
print("=" * 100)

print("\n✂ Splitting dataset...")

X_train_text, X_test_text, y_train_raw, y_test_raw = train_test_split(
    df['cleaned'],
    df['type'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df['type']
)

print(f"✔ Training samples: {len(X_train_text):,}")
print(f"✔ Testing samples : {len(X_test_text):,}")
print(f"✔ Train/Test ratio: {len(X_train_text)/len(X_test_text):.1f}:1")

# =============================================================================
# PHASE 4 · FEATURE EXTRACTION (TF-IDF)
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 4 · FEATURE EXTRACTION")
print("=" * 100)

print("\n🔠 TF-IDF Vectorization...")
print(f"   Max features: {MAX_FEATURES:,}")
print(f"   N-gram range: {NGRAM_RANGE}")

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    min_df=2,           # Minimum document frequency
    max_df=0.95,        # Maximum document frequency
    stop_words="english",
    sublinear_tf=True,  # Apply sublinear TF scaling
    use_idf=True,
    norm='l2'
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"\n✔ Training features: {X_train.shape}")
print(f"✔ Testing features : {X_test.shape}")
print(f"✔ Feature sparsity : {(1.0 - (X_train.nnz / (X_train.shape[0] * X_train.shape[1]))) * 100:.1f}%")

# Label encoding
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

print("\n🏷 Label Mapping:")
for i, label in enumerate(le.classes_):
    print(f"  {i:2d} → {label}")

print(f"\n✔ Total classes: {len(le.classes_)}")

# =============================================================================
# PHASE 5 · MODEL TRAINING
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 5 · MODEL TRAINING")
print("=" * 100)

results = {}

# =========================================================================
# 1. LOGISTIC REGRESSION
# =========================================================================

print("\n[1/5] Logistic Regression")
print("-" * 50)

start = time.time()

lr_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver='lbfgs',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)

lr_acc = accuracy_score(y_test, lr_pred)
lr_bal = balanced_accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred, average='weighted')
lr_prec = precision_score(y_test, lr_pred, average='weighted')
lr_rec = recall_score(y_test, lr_pred, average='weighted')

end = time.time()

results["Logistic Regression"] = {
    "model": lr_model,
    "pred": lr_pred,
    "proba": lr_proba,
    "accuracy": lr_acc,
    "balanced_acc": lr_bal,
    "f1": lr_f1,
    "precision": lr_prec,
    "recall": lr_rec,
    "time": end - start
}

print(f"✔ Accuracy        : {lr_acc*100:6.2f}%")
print(f"✔ Balanced Acc    : {lr_bal*100:6.2f}%")
print(f"✔ F1 Score        : {lr_f1*100:6.2f}%")
print(f"✔ Precision       : {lr_prec*100:6.2f}%")
print(f"✔ Recall          : {lr_rec*100:6.2f}%")
print(f"✔ Training time   : {(end-start):.2f} sec")

# =========================================================================
# 2. MULTINOMIAL NAIVE BAYES
# =========================================================================

print("\n[2/5] Multinomial Naive Bayes")
print("-" * 50)

start = time.time()

nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
nb_proba = nb_model.predict_proba(X_test)

nb_acc = accuracy_score(y_test, nb_pred)
nb_bal = balanced_accuracy_score(y_test, nb_pred)
nb_f1 = f1_score(y_test, nb_pred, average='weighted')
nb_prec = precision_score(y_test, nb_pred, average='weighted')
nb_rec = recall_score(y_test, nb_pred, average='weighted')

end = time.time()

results["Naive Bayes"] = {
    "model": nb_model,
    "pred": nb_pred,
    "proba": nb_proba,
    "accuracy": nb_acc,
    "balanced_acc": nb_bal,
    "f1": nb_f1,
    "precision": nb_prec,
    "recall": nb_rec,
    "time": end - start
}

print(f"✔ Accuracy        : {nb_acc*100:6.2f}%")
print(f"✔ Balanced Acc    : {nb_bal*100:6.2f}%")
print(f"✔ F1 Score        : {nb_f1*100:6.2f}%")
print(f"✔ Precision       : {nb_prec*100:6.2f}%")
print(f"✔ Recall          : {nb_rec*100:6.2f}%")
print(f"✔ Training time   : {(end-start):.2f} sec")

# =========================================================================
# 3. RANDOM FOREST
# =========================================================================

print("\n[3/5] Random Forest")
print("-" * 50)

start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
rf_bal = balanced_accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')
rf_prec = precision_score(y_test, rf_pred, average='weighted')
rf_rec = recall_score(y_test, rf_pred, average='weighted')

end = time.time()

results["Random Forest"] = {
    "model": rf_model,
    "pred": rf_pred,
    "proba": rf_proba,
    "accuracy": rf_acc,
    "balanced_acc": rf_bal,
    "f1": rf_f1,
    "precision": rf_prec,
    "recall": rf_rec,
    "time": end - start
}

print(f"✔ Accuracy        : {rf_acc*100:6.2f}%")
print(f"✔ Balanced Acc    : {rf_bal*100:6.2f}%")
print(f"✔ F1 Score        : {rf_f1*100:6.2f}%")
print(f"✔ Precision       : {rf_prec*100:6.2f}%")
print(f"✔ Recall          : {rf_rec*100:6.2f}%")
print(f"✔ Training time   : {(end-start):.2f} sec")

# =========================================================================
# 4. LINEAR SVM
# =========================================================================

print("\n[4/5] Linear SVM")
print("-" * 50)

start = time.time()

svm_model = LinearSVC(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    verbose=0
)

svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
svm_decision = svm_model.decision_function(X_test)
# Convert decision scores to probabilities
svm_proba = np.exp(svm_decision) / np.exp(svm_decision).sum(axis=1, keepdims=True)

svm_acc = accuracy_score(y_test, svm_pred)
svm_bal = balanced_accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average='weighted')
svm_prec = precision_score(y_test, svm_pred, average='weighted')
svm_rec = recall_score(y_test, svm_pred, average='weighted')

end = time.time()

results["Linear SVM"] = {
    "model": svm_model,
    "pred": svm_pred,
    "proba": svm_proba,
    "accuracy": svm_acc,
    "balanced_acc": svm_bal,
    "f1": svm_f1,
    "precision": svm_prec,
    "recall": svm_rec,
    "time": end - start
}

print(f"✔ Accuracy        : {svm_acc*100:6.2f}%")
print(f"✔ Balanced Acc    : {svm_bal*100:6.2f}%")
print(f"✔ F1 Score        : {svm_f1*100:6.2f}%")
print(f"✔ Precision       : {svm_prec*100:6.2f}%")
print(f"✔ Recall          : {svm_rec*100:6.2f}%")
print(f"✔ Training time   : {(end-start):.2f} sec")

# =========================================================================
# 5. XGBOOST (Optional)
# =========================================================================

if HAS_XGB:
    print("\n[5/5] XGBoost")
    print("-" * 50)

    start = time.time()

    sample_weights = compute_sample_weight("balanced", y=y_train)

    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=RANDOM_STATE,
        eval_metric="mlogloss",
        verbosity=0,
        n_jobs=4
    )

    xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
    xgb_pred = xgb_model.predict(X_test)
    xgb_proba = xgb_model.predict_proba(X_test)

    xgb_acc = accuracy_score(y_test, xgb_pred)
    xgb_bal = balanced_accuracy_score(y_test, xgb_pred)
    xgb_f1 = f1_score(y_test, xgb_pred, average='weighted')
    xgb_prec = precision_score(y_test, xgb_pred, average='weighted')
    xgb_rec = recall_score(y_test, xgb_pred, average='weighted')

    end = time.time()

    results["XGBoost"] = {
        "model": xgb_model,
        "pred": xgb_pred,
        "proba": xgb_proba,
        "accuracy": xgb_acc,
        "balanced_acc": xgb_bal,
        "f1": xgb_f1,
        "precision": xgb_prec,
        "recall": xgb_rec,
        "time": end - start
    }

    print(f"✔ Accuracy        : {xgb_acc*100:6.2f}%")
    print(f"✔ Balanced Acc    : {xgb_bal*100:6.2f}%")
    print(f"✔ F1 Score        : {xgb_f1*100:6.2f}%")
    print(f"✔ Precision       : {xgb_prec*100:6.2f}%")
    print(f"✔ Recall          : {xgb_rec*100:6.2f}%")
    print(f"✔ Training time   : {(end-start):.2f} sec")

else:
    print("\n⚠ Skipping XGBoost (not installed)")

# =========================================================================
# 6. WEIGHTED ENSEMBLE
# =========================================================================

print("\n[6/6] Weighted Ensemble")
print("-" * 50)

# Calculate ensemble predictions with weights
if HAS_XGB:
    # All 5 models
    ens_proba = (
        0.25 * lr_proba +      # Logistic Regression
        0.15 * nb_proba +      # Naive Bayes
        0.20 * rf_proba +      # Random Forest
        0.20 * svm_proba +     # SVM
        0.20 * xgb_proba       # XGBoost
    )
else:
    # 4 models
    ens_proba = (
        0.30 * lr_proba +
        0.20 * nb_proba +
        0.25 * rf_proba +
        0.25 * svm_proba
    )

ens_pred = np.argmax(ens_proba, axis=1)
ens_acc = accuracy_score(y_test, ens_pred)
ens_bal = balanced_accuracy_score(y_test, ens_pred)
ens_f1 = f1_score(y_test, ens_pred, average='weighted')
ens_prec = precision_score(y_test, ens_pred, average='weighted')
ens_rec = recall_score(y_test, ens_pred, average='weighted')

results["Ensemble"] = {
    "model": None,
    "pred": ens_pred,
    "proba": ens_proba,
    "accuracy": ens_acc,
    "balanced_acc": ens_bal,
    "f1": ens_f1,
    "precision": ens_prec,
    "recall": ens_rec,
    "time": 0
}

print(f"✔ Accuracy        : {ens_acc*100:6.2f}%")
print(f"✔ Balanced Acc    : {ens_bal*100:6.2f}%")
print(f"✔ F1 Score        : {ens_f1*100:6.2f}%")
print(f"✔ Precision       : {ens_prec*100:6.2f}%")
print(f"✔ Recall          : {ens_rec*100:6.2f}%")

# =============================================================================
# PHASE 6 · RESULTS ANALYSIS
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 6 · RESULTS ANALYSIS")
print("=" * 100)

# Find best model
best_model_name = max(results, key=lambda x: results[x]["accuracy"])
best_result = results[best_model_name]
best_acc = best_result["accuracy"]
best_pred = best_result["pred"]

print(f"\n🏆 Best Model: {best_model_name}")
print(f"✔ Accuracy: {best_acc*100:.2f}%")

status = "✅ TARGET ACHIEVED" if best_acc >= ACCURACY_TARGET else "⚠ BELOW TARGET"
print(f"✔ Status: {status}")

# Model comparison table
print("\n" + "-" * 100)
print(f"{'MODEL':<20} {'ACCURACY':<12} {'BALANCED':<12} {'F1 SCORE':<12} {'PRECISION':<12} {'RECALL':<12}")
print("-" * 100)

for name in sorted(results.keys(), key=lambda x: results[x]["accuracy"], reverse=True):
    result = results[name]
    print(
        f"{name:<20} "
        f"{result['accuracy']*100:>10.2f}% "
        f"{result['balanced_acc']*100:>10.2f}% "
        f"{result['f1']*100:>10.2f}% "
        f"{result['precision']*100:>10.2f}% "
        f"{result['recall']*100:>10.2f}%"
    )

# Classification report
print("\n" + "=" * 100)
print("DETAILED CLASSIFICATION REPORT (Best Model)")
print("=" * 100)
print("\n" + classification_report(y_test, best_pred, target_names=le.classes_))

# =============================================================================
# PHASE 7 · VISUALIZATIONS
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 7 · GENERATING VISUALIZATIONS")
print("=" * 100)

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------------------------------------------------
# 1. Confusion Matrix
# -------------------------------------------------------------------------

print("\n📉 Creating Confusion Matrix...")

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(16, 14))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    cbar_kws={"label": "Count"}
)

plt.title(f"Confusion Matrix - {best_model_name} (Accuracy: {best_acc*100:.2f}%)", fontsize=16, fontweight='bold')
plt.xlabel("Predicted Type", fontsize=12, fontweight='bold')
plt.ylabel("True Type", fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(f"{OUTPUT_DIR}/01_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.close()

print("✔ Saved: 01_confusion_matrix.png")

# -------------------------------------------------------------------------
# 2. Model Accuracy Comparison
# -------------------------------------------------------------------------

print("\n📊 Creating Model Comparison...")

model_names = list(results.keys())
accuracies = [results[m]["accuracy"] * 100 for m in model_names]
f1_scores_list = [results[m]["f1"] * 100 for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
ax1 = axes[0]
colors = ['#2ecc71' if acc >= ACCURACY_TARGET * 100 else '#e74c3c' for acc in accuracies]
bars1 = ax1.bar(model_names, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.axhline(y=ACCURACY_TARGET * 100, color='red', linestyle='--', linewidth=2, label=f'Target ({ACCURACY_TARGET*100:.0f}%)')
ax1.set_ylabel("Accuracy (%)", fontsize=12, fontweight='bold')
ax1.set_title("Model Accuracy Comparison", fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars1, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height + 2, f'{acc:.1f}%',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

ax1.set_xticklabels(model_names, rotation=45, ha='right')

# F1 Score comparison
ax2 = axes[1]
bars2 = ax2.bar(model_names, f1_scores_list, color='#3498db', alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_ylabel("F1 Score (%)", fontsize=12, fontweight='bold')
ax2.set_title("Model F1 Score Comparison", fontsize=14, fontweight='bold')
ax2.set_ylim(0, 100)
ax2.grid(axis='y', alpha=0.3)

for bar, f1 in zip(bars2, f1_scores_list):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 2, f'{f1:.1f}%',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.set_xticklabels(model_names, rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_model_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

print("✔ Saved: 02_model_comparison.png")

# -------------------------------------------------------------------------
# 3. Per-Class F1 Scores
# -------------------------------------------------------------------------

print("\n📈 Creating Per-Class F1 Scores...")

f1_scores_per_class = f1_score(y_test, best_pred, average=None)

sorted_idx = np.argsort(f1_scores_per_class)[::-1]
sorted_classes = np.array(le.classes_)[sorted_idx]
sorted_scores = f1_scores_per_class[sorted_idx]

plt.figure(figsize=(12, 8))

bars = plt.bar(sorted_classes, sorted_scores, color='#9b59b6', alpha=0.7, edgecolor='black', linewidth=1.5)

plt.ylim(0, 1)
plt.ylabel("F1 Score", fontsize=12, fontweight='bold')
plt.title(f"Per-Class F1 Scores ({best_model_name})", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

for bar, score in zip(bars, sorted_scores):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{score:.2f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_per_class_f1.png", dpi=300, bbox_inches='tight')
plt.close()

print("✔ Saved: 03_per_class_f1.png")

# -------------------------------------------------------------------------
# 4. Precision, Recall, F1 Comparison
# -------------------------------------------------------------------------

print("\n📊 Creating Metrics Comparison...")

metrics_names = list(results.keys())
accuracies_vals = [results[m]["accuracy"] * 100 for m in metrics_names]
precisions_vals = [results[m]["precision"] * 100 for m in metrics_names]
recalls_vals = [results[m]["recall"] * 100 for m in metrics_names]
f1_vals = [results[m]["f1"] * 100 for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 7))

ax.bar(x - 1.5*width, accuracies_vals, width, label='Accuracy', alpha=0.8)
ax.bar(x - 0.5*width, precisions_vals, width, label='Precision', alpha=0.8)
ax.bar(x + 0.5*width, recalls_vals, width, label='Recall', alpha=0.8)
ax.bar(x + 1.5*width, f1_vals, width, label='F1 Score', alpha=0.8)

ax.set_ylabel("Score (%)", fontsize=12, fontweight='bold')
ax.set_title("Model Metrics Comparison", fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_metrics_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

print("✔ Saved: 04_metrics_comparison.png")

# -------------------------------------------------------------------------
# 5. Type Distribution
# -------------------------------------------------------------------------

print("\n📊 Creating Type Distribution Visualization...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Train distribution
train_dist = pd.Series(y_train_raw).value_counts()
ax1 = axes[0]
ax1.barh(train_dist.index, train_dist.values, color='#3498db', alpha=0.7, edgecolor='black')
ax1.set_xlabel("Count", fontsize=12, fontweight='bold')
ax1.set_title("Training Set Distribution", fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Test distribution
test_dist = pd.Series(y_test_raw).value_counts()
ax2 = axes[1]
ax2.barh(test_dist.index, test_dist.values, color='#e74c3c', alpha=0.7, edgecolor='black')
ax2.set_xlabel("Count", fontsize=12, fontweight='bold')
ax2.set_title("Test Set Distribution", fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_type_distribution.png", dpi=300, bbox_inches='tight')
plt.close()

print("✔ Saved: 05_type_distribution.png")

# =============================================================================
# PHASE 8 · SAVE MODELS & ARTIFACTS
# =============================================================================

print("\n" + "=" * 100)
print("PHASE 8 · SAVING ARTIFACTS")
print("=" * 100)

artifacts = {
    "best_model_name": best_model_name,
    "best_accuracy": best_acc,
    "label_encoder": le,
    "vectorizer": vectorizer,
    "logistic_regression": results["Logistic Regression"]["model"],
    "naive_bayes": results["Naive Bayes"]["model"],
    "random_forest": results["Random Forest"]["model"],
    "svm": results["Linear SVM"]["model"],
    "xgboost": results.get("XGBoost", {}).get("model"),
    "ensemble_weights": {
        "logistic_regression": 0.25,
        "naive_bayes": 0.15,
        "random_forest": 0.20,
        "svm": 0.20,
        "xgboost": 0.20 if HAS_XGB else 0
    }
}

save_path = f"{OUTPUT_DIR}/mbti_complete_model.pkl"

with open(save_path, "wb") as f:
    pickle.dump(artifacts, f)

size_mb = os.path.getsize(save_path) / (1024 * 1024)

print(f"\n✔ Model artifacts saved")
print(f"  File: {save_path}")
print(f"  Size: {size_mb:.2f} MB")

def predict_personality(text):
    """
    Predict MBTI personality type for new text, handling both 
    single models and the weighted ensemble.
    """
    # 1. Preprocess & Vectorize
    cleaned = preprocess_text(text)
    X_new = vectorizer.transform([cleaned])
    
    # 2. Check if we are using the Ensemble or a single model
    if best_model_name == "Ensemble":
        # Define the models and weights used in your Phase 5
        # (Ensure these names match the keys in your 'results' dictionary)
        model_names = ["Logistic Regression", "Multinomial Naive Bayes", "Random Forest", "Linear SVM", "XGBoost"]
        weights = [0.30, 0.05, 0.15, 0.20, 0.30] # Adjust these to match your actual Phase 5 weights
        
        all_probs = []
        for name in model_names:
            model = results[name]["model"]
            # Get probabilities from each individual model
            all_probs.append(model.predict_proba(X_new)[0])
        
        # Calculate weighted average probabilities
        probabilities = np.average(all_probs, axis=0, weights=weights)
        prediction = np.argmax(probabilities)
        
    else:
        # Standard logic for a single model (Logistic Regression, XGBoost, etc.)
        best_model = results[best_model_name]["model"]
        prediction = best_model.predict(X_new)[0]
        probabilities = best_model.predict_proba(X_new)[0]
    
    # 3. Get top 3 predictions for the UI/Output
    top_3_idx = np.argsort(probabilities)[::-1][:3]
    
    return {
        "predicted_type": le.classes_[prediction],
        "confidence": probabilities[prediction] * 100,
        "top_3": [
            {
                "type": le.classes_[idx],
                "probability": probabilities[idx] * 100
            }
            for idx in top_3_idx
        ]
    }

# Test examples
test_examples = [
    "I love meeting new people and exploring different cultures",
    "I prefer to stay home and read books rather than go out",
    "I always plan everything carefully and follow the rules",
    "I'm very creative and hate following structured routines",
    "I enjoy analyzing things deeply and thinking about abstract concepts"
]

print("\n🔮 Sample Predictions:\n")

for i, text in enumerate(test_examples, 1):
    result = predict_personality(text)
    
    print(f"{i}. Text: {text[:60]}...")
    print(f"   Predicted Type: {result['predicted_type']}")
    print(f"   Confidence: {result['confidence']:.2f}%")
    print(f"   Top 3:")
    for j, pred in enumerate(result['top_3'], 1):
        print(f"      {j}. {pred['type']}: {pred['probability']:.2f}%")
    print()

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("✅ TRAINING COMPLETE - SUMMARY")
print("=" * 100)

summary = f"""
DATASET STATISTICS
──────────────────
  Total samples    : {len(df):,}
  Training samples : {len(X_train_text):,}
  Testing samples  : {len(X_test_text):,}
  Total classes    : {len(le.classes_)}
  Class balance    : {'Good' if min(dist.values)/max(dist.values) > 0.2 else 'Imbalanced'}

PREPROCESSING
──────────────────
  Text cleaning steps: 11 (URLs, emails, emojis, punctuation, stopwords, etc.)
  Average text length: {lengths.mean():.0f} words
  Min/Max words      : {lengths.min()} / {lengths.max()}

FEATURE EXTRACTION
──────────────────
  Vectorizer      : TF-IDF
  Max features    : {MAX_FEATURES:,}
  Sparsity        : {(1.0 - (X_train.nnz / (X_train.shape[0] * X_train.shape[1]))) * 100:.1f}%
  N-gram range    : {NGRAM_RANGE}

MODELS TRAINED
──────────────────
  1. Logistic Regression : {results['Logistic Regression']['accuracy']*100:.2f}%
  2. Naive Bayes         : {results['Naive Bayes']['accuracy']*100:.2f}%
  3. Random Forest       : {results['Random Forest']['accuracy']*100:.2f}%
  4. Linear SVM          : {results['Linear SVM']['accuracy']*100:.2f}%
  {f'5. XGBoost             : {results["XGBoost"]["accuracy"]*100:.2f}%' if HAS_XGB else ''}
  6. Ensemble            : {results['Ensemble']['accuracy']*100:.2f}%

BEST MODEL
──────────────────
  Model            : {best_model_name}
  Accuracy         : {best_acc*100:.2f}%
  Balanced Accuracy: {results[best_model_name]['balanced_acc']*100:.2f}%
  F1 Score         : {results[best_model_name]['f1']*100:.2f}%
  Precision        : {results[best_model_name]['precision']*100:.2f}%
  Recall           : {results[best_model_name]['recall']*100:.2f}%
  Status           : {status}

OUTPUT FILES GENERATED
──────────────────────
  ✓ 01_confusion_matrix.png
  ✓ 02_model_comparison.png
  ✓ 03_per_class_f1.png
  ✓ 04_metrics_comparison.png
  ✓ 05_type_distribution.png
  ✓ mbti_complete_model.pkl

NEXT STEPS
──────────────────
  1. Try hyperparameter tuning for better accuracy
  2. Experiment with Deep Learning (DistilBERT, LSTM)
  3. Combine with other datasets for larger training set
  4. Deploy with Streamlit or Flask
  5. Fine-tune for production use

Thank you for using this comprehensive MBTI pipeline! 🚀
"""

print(summary)

print("\n" + "=" * 100)
print("Script completed successfully!")
print("=" * 100)



                    COMPLETE MBTI PERSONALITY PREDICTION PIPELINE

This pipeline includes:
  ✓ Data loading & validation
  ✓ Advanced text preprocessing
  ✓ Feature extraction (TF-IDF)
  ✓ Model training (5 models)
  ✓ Comprehensive evaluation
  ✓ Visualizations
  ✓ Prediction on new text


PHASE 1 · LOAD DATASET

📥 Loading dataset from:
   C:\Users\Ateeb\-MBTI-Myers-Briggs-Personality-Type-Dataset-master\-MBTI-Myers-Briggs-Personality-Type-Dataset-master/mbti_1.csv

✔ Dataset downloaded successfully!

✔ Total samples: 8,675
✔ Columns: ['type', 'posts']

First 3 rows:
   type                                              posts
0  INFJ  'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1  ENTP  'I'm finding the lack of me in these posts ver...
2  INTP  'Good one  _____   https://www.youtube.com/wat...

✔ Memory usage: 60.04 MB
✔ After removing nulls: 8,675 samples

📊 MBTI Type Distribution:

  INFP    1832 ( 21.1%) ██████████
  INFJ    1470 ( 16.9%) ████████
  INTP    1304 ( 15.0%) ████

KeyError: 'Multinomial Naive Bayes'

In [21]:
import joblib
import numpy as np

# 1. Load the artifacts you already saved in Phase 8
# This avoids the 30-minute training wait!
artifacts = joblib.load('./mbti_complete_model.pkl')

# Extract what we need
# Note: Ensure your Phase 8 save included these keys
vectorizer = artifacts['vectorizer']
le = artifacts['label_encoder']
results = artifacts['results'] 
best_model_name = artifacts['best_model_name']

def predict_personality(text):
    cleaned = preprocess_text(text) # Ensure this function is defined in your current session
    X_new = vectorizer.transform([cleaned])
    
    if best_model_name == "Ensemble":
        # Match these keys EXACTLY to what is in your results dictionary
        # Based on your logs, try "Naive Bayes" instead of "Multinomial Naive Bayes"
        model_names = ["Logistic Regression", "Naive Bayes", "Random Forest", "Linear SVM", "XGBoost"]
        weights = [0.30, 0.05, 0.15, 0.20, 0.30] 
        
        all_probs = []
        for name in model_names:
            if name in results:
                model = results[name]["model"]
                all_probs.append(model.predict_proba(X_new)[0])
            else:
                print(f"Warning: {name} not found in results keys: {results.keys()}")
        
        probabilities = np.average(all_probs, axis=0, weights=weights)
        prediction = np.argmax(probabilities)
    else:
        best_model = results[best_model_name]["model"]
        prediction = best_model.predict(X_new)[0]
        probabilities = best_model.predict_proba(X_new)[0]

    return {
        "predicted_type": le.classes_[prediction],
        "confidence": probabilities[prediction] * 100,
        "top_3": [
            {"type": le.classes_[idx], "probability": probabilities[idx] * 100}
            for idx in np.argsort(probabilities)[::-1][:3]
        ]
    }

# Now try your sample predictions
test_text = "I enjoy over-analyzing movie plots and spending time alone to recharge."
print(predict_personality(test_text))

KeyError: 'results'

In [2]:
# =============================================================================
# MINIMAL MBTI PIPELINE (Clean & Deployment‑Ready)
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import re
import string
import pickle
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# Download stopwords once
nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

# -----------------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------------
CSV_PATH = r"C:\Users\Ateeb\-MBTI-Myers-Briggs-Personality-Type-Dataset-master\-MBTI-Myers-Briggs-Personality-Type-Dataset-master/mbti_1.csv"
# Change to your file path
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_FEATURES = 5000
NGRAM_RANGE = (1, 2)

# -----------------------------------------------------------------------------
# 2. TEXT PREPROCESSING
# -----------------------------------------------------------------------------
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("|||", " ").lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return ' '.join(words)

# -----------------------------------------------------------------------------
# 3. LOAD & PREPROCESS DATA
# -----------------------------------------------------------------------------
print("Loading data...")
df = pd.read_csv(CSV_PATH, encoding='latin1')
df = df.dropna(subset=['type', 'posts']).reset_index(drop=True)
print(f"Initial samples: {len(df)}")

print("Cleaning text...")
df['cleaned'] = df['posts'].apply(preprocess_text)
df = df[df['cleaned'].str.len() > 0].reset_index(drop=True)
print(f"After cleaning: {len(df)}")

# -----------------------------------------------------------------------------
# 4. TRAIN/TEST SPLIT & TF‑IDF
# -----------------------------------------------------------------------------
X_train_text, X_test_text, y_train_raw, y_test_raw = train_test_split(
    df['cleaned'], df['type'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df['type']
)

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    min_df=2,
    max_df=0.95,
    stop_words='english',
    sublinear_tf=True
)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

# -----------------------------------------------------------------------------
# 5. TRAIN MODELS
# -----------------------------------------------------------------------------
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1)
lr.fit(X_train, y_train)

models = {'LogisticRegression': lr}
if HAS_XGB:
    print("Training XGBoost...")
    sample_weight = compute_sample_weight('balanced', y_train)
    xgb = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE)
    xgb.fit(X_train, y_train, sample_weight=sample_weight)
    models['XGBoost'] = xgb

# -----------------------------------------------------------------------------
# 6. ENSEMBLE PREDICTION (weighted)
# -----------------------------------------------------------------------------
def ensemble_predict_proba(X):
    proba_list = []
    if 'LogisticRegression' in models:
        proba_list.append(0.5 * models['LogisticRegression'].predict_proba(X))
    if 'XGBoost' in models:
        proba_list.append(0.5 * models['XGBoost'].predict_proba(X))
    return np.average(proba_list, axis=0)

y_pred = np.argmax(ensemble_predict_proba(X_test), axis=1)
acc = accuracy_score(y_test, y_pred)
print(f"\nEnsemble Accuracy: {acc*100:.2f}%")

# (Optional) Quick confusion matrix – you can remove if not needed
import matplotlib.pyplot as plt
import seaborn as sns
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix (Acc: {acc*100:.1f}%)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=200)
plt.close()
print("Saved confusion_matrix.png")

# -----------------------------------------------------------------------------
# 7. SAVE EVERYTHING FOR DEPLOYMENT
# -----------------------------------------------------------------------------
artifacts = {
    'vectorizer': vectorizer,
    'label_encoder': le,
    'logistic_regression': lr,
    'xgboost': xgb if HAS_XGB else None,
    'ensemble_weights': (0.5, 0.5) if HAS_XGB else (1.0,)
}

with open('mbti_model.pkl', 'wb') as f:
    pickle.dump(artifacts, f)
print("Saved model to mbti_model.pkl")

# -----------------------------------------------------------------------------
# 8. PREDICTION FUNCTION (used later by API/UI)
# -----------------------------------------------------------------------------
def predict_mbti(text):
    cleaned = preprocess_text(text)
    X = artifacts['vectorizer'].transform([cleaned])
    proba = []
    if artifacts['logistic_regression']:
        proba.append(0.5 * artifacts['logistic_regression'].predict_proba(X))
    if artifacts['xgboost']:
        proba.append(0.5 * artifacts['xgboost'].predict_proba(X))
    final_proba = np.average(proba, axis=0)[0]
    pred_idx = np.argmax(final_proba)
    mbti_type = artifacts['label_encoder'].classes_[pred_idx]
    confidence = final_proba[pred_idx] * 100
    return mbti_type, confidence

# Quick test (you can comment out)
test_text = "I love planning everything ahead and following rules"
pred_type, conf = predict_mbti(test_text)
print(f"\nTest prediction: {pred_type} ({conf:.1f}%)")

Loading data...
Initial samples: 8675
Cleaning text...
After cleaning: 8674
Training Logistic Regression...
Training XGBoost...

Ensemble Accuracy: 68.01%
Saved confusion_matrix.png
Saved model to mbti_model.pkl

Test prediction: INTP (6.2%)


In [5]:
import pickle
import re
import string
import numpy as np
from nltk.corpus import stopwords
import nltk

# Ensure stopwords are available
nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

# ----------------------------------------------------------------------
# Text preprocessing (must match training pipeline)
# ----------------------------------------------------------------------
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("|||", " ").lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return ' '.join(words)

# ----------------------------------------------------------------------
# Load the saved model
# ----------------------------------------------------------------------
with open('mbti_model.pkl', 'rb') as f:
    artifacts = pickle.load(f)

vectorizer = artifacts['vectorizer']
label_encoder = artifacts['label_encoder']
lr_model = artifacts['logistic_regression']
xgb_model = artifacts.get('xgboost')  # may be None if XGBoost wasn't saved

# ----------------------------------------------------------------------
# Prediction function
# ----------------------------------------------------------------------
def predict_mbti(text):
    cleaned = preprocess_text(text)
    X = vectorizer.transform([cleaned])
    
    proba_list = []
    if lr_model is not None:
        proba_list.append(0.5 * lr_model.predict_proba(X))
    if xgb_model is not None:
        proba_list.append(0.5 * xgb_model.predict_proba(X))
    
    final_proba = np.average(proba_list, axis=0)[0]
    pred_idx = np.argmax(final_proba)
    mbti_type = label_encoder.classes_[pred_idx]
    confidence = final_proba[pred_idx] * 100
    
    # Get top 3 predictions
    top3_idx = np.argsort(final_proba)[::-1][:3]
    top3 = [(label_encoder.classes_[i], final_proba[i] * 100) for i in top3_idx]
    
    return {
        'predicted_type': mbti_type,
        'confidence': confidence,
        'top_3': top3
    }

# ----------------------------------------------------------------------
# Test paragraph (INFP style)
# ----------------------------------------------------------------------
test_paragraph = """I've always felt different from most people. While others seem happy following routines and social norms, I find myself constantly questioning the meaning behind everything. Why do we work 9 to 5? Why do people care so much about status? I spend hours lost in my own thoughts, imagining possibilities – what if society was built around creativity instead of money? I love writing poetry and having deep conversations about philosophy late at night. Small talk drains me, but I could discuss human emotions or fictional worlds for hours. I'm very sensitive to criticism, even when it's constructive. I want my work to reflect my values – helping others in a meaningful way. I often procrastinate because I wait for the 'perfect moment' of inspiration, which rarely comes. My room is messy, but my ideas are organized. I believe everyone has a unique inner world that deserves to be understood."""

# Run prediction
result = predict_mbti(test_paragraph)

print("\n" + "="*60)
print("MBTI PREDICTION RESULT")
print("="*60)
print(f"\n📝 Text length: {len(test_paragraph)} characters, ~{len(test_paragraph.split())} words\n")
print(f"🎯 Predicted MBTI: {result['predicted_type']}")
print(f"📊 Confidence: {result['confidence']:.1f}%")
print(f"\n🏆 Top 3 types:")
for i, (t, prob) in enumerate(result['top_3'], 1):
    bar = "█" * int(prob/2)
    print(f"   {i}. {t}: {prob:.1f}%  {bar}")
print("\n" + "="*60)


MBTI PREDICTION RESULT

📝 Text length: 892 characters, ~150 words

🎯 Predicted MBTI: INFP
📊 Confidence: 12.1%

🏆 Top 3 types:
   1. INFP: 12.1%  ██████
   2. INTP: 7.7%  ███
   3. INTJ: 5.4%  ██



In [6]:
pip install streamlit


   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 1.8 MB/s eta 0:00:06
   - -------------------------------------- 0.5/11.0 MB 1.8 MB/s eta 0:00:06
   -- ------------------------------------- 0.8/11.0 MB 993.6 kB/s eta 0:00:11
   -- ------------------------------------- 0.8/11.0 MB 993.6 kB/s eta 0:00:11
   --- ------------------------------------ 1.0/11.0 MB 836.9 kB/s eta 0:00:12
   ---- ----------------------------------- 1.3/11.0 MB 990.9 kB/s eta 0:00:10
   ---- ----------------------------------- 1.3/11.0 MB 990.9 kB/s eta 0:00:10
   ------ --------------------------------- 1.8/11.0 MB 965.2 kB/s eta 0:00:10
   ------ --------------------------------- 1.8/11.0 MB 965.2 kB/s eta 0:00:10
   ------- -------------------------------- 2.1/11.0 MB 938.4 kB/s eta 0:00:10
   

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [8]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)